# SDK Analysis (optional deep dive)

Interactive charts and raw request/response pairs from SDK DEBUG hooks in **`TF_LOG=json`**
captures. Works with export, plan, or apply logs.

For structured summaries (verdicts, timelines, issue attribution), run the matching **hang** or
**performance** notebook and export **`*-report.json`** — see **`HOW-TO-READ-RESULTS.md`**. Use
this notebook when you need raw pairs or ad-hoc endpoint charts.

Capture example (plan):

```bash
export TF_LOG=json
export TF_LOG_PATH=plan-tflog.log
terraform plan
export TERRAFORM_LOG_PATH=plan-tflog.log
```

For response-time percentiles by endpoint, use **`log-chomper/`**. Run `whatisit.ipynb` if unsure.


In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import pandas as pd
import commonlib.prep_sdk_data as prep_sdk_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg


In [ ]:
c = cfg.Config()
print(f"Reading terraform data from: {c.TERRAFORM_LOG_PATH}")
normalized_records = prep_sdk_data.load_normalized_records()
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No SDK DEBUG records found. Capture with TF_LOG=json during plan/apply/export. "
        "Confirm the file with whatisit.ipynb."
    )

print(sorted(df["debug_type"].drop_duplicates().tolist()))

df_sdk_request = df[df["debug_type"] == "SDK DEBUG REQUEST"].rename(
    columns={"timestamp": "request_timestamp"}
)
df_sdk_response = df[df["debug_type"] == "SDK DEBUG RESPONSE"].rename(
    columns={"timestamp": "response_timestamp"}
)

df_sdk_request_response = pd.merge(
    df_sdk_request[
        ["transaction_id", "invocation_method", "invocation_url", "sanitized_url", "request_timestamp"]
    ],
    df_sdk_response[
        ["transaction_id", "response_timestamp", "invocation_status_code", "invocation_retry_after"]
    ],
    on="transaction_id",
)

df_sdk_request_response["method_url"] = df_sdk_request_response.apply(
    lambda row: f"{row['invocation_method']} {row['sanitized_url']}", axis=1
)

## SDK 429 rate-limit wait time

Summed `invocation_retry_after` from **429** responses — estimated time the SDK was told to wait on rate limits.


In [ ]:
df_429 = df_sdk_request_response[df_sdk_request_response["invocation_status_code"] == 429].copy()
wait_total = int(df_429["invocation_retry_after"].fillna(0).astype(int).sum())
if wait_total:
    wait_minutes = wait_total / 60
    print(
        f"429 wait time: {wait_minutes:.1f} min across {len(df_429):,} rate-limit responses"
    )
    by_endpoint = (
        df_429.groupby("method_url")["invocation_retry_after"]
        .agg(response_429="count", wait_seconds="sum")
        .reset_index()
    )
    by_endpoint["wait_minutes"] = (by_endpoint["wait_seconds"] / 60).round(2)
    by_endpoint = by_endpoint.sort_values("wait_seconds", ascending=False)
    display(by_endpoint.head(20))
    plot_df = by_endpoint.head(15).sort_values("wait_minutes")
    plt.figure(figsize=(12, 6))
    plt.barh(plot_df["method_url"], plot_df["wait_minutes"], color="tab:orange")
    plt.xlabel("wait minutes (summed retry_after)")
    plt.title("SDK 429 rate-limit wait time by endpoint")
    plt.tight_layout()
else:
    print("No SDK 429 responses (or no invocation_retry_after on 429 lines).")


## API Call Volume

In [ ]:
gencharts.generate_plt_by_method_url(
    df_sdk_request_response,
    df_sdk_request_response["method_url"],
    top_n=20,
)

## SDK API volume over time

Requests per **minute from log start** (same view as hang/performance notebooks). Highlights when DNC list export polling starts and peaks.

In [ ]:
df_sdk_timeline = prep_sdk_data.sdk_requests_timeline_dataframe(normalized_records, bucket_minutes=1)
if df_sdk_timeline.empty:
    print("No SDK request timeline data.")
else:
    export_rows = df_sdk_timeline[df_sdk_timeline["is_dnclist_export"]]
    if not export_rows.empty:
        first_min = export_rows["minute_from_start"].min()
        peak = export_rows.groupby("minute_from_start")["request_count"].sum().idxmax()
        peak_count = export_rows.groupby("minute_from_start")["request_count"].sum().max()
        print(f"First DNC list export activity: minute {first_min:.1f} from log start")
        print(f"DNC list export peak: {peak_count} req/min at minute {peak:.1f}")
    gencharts.plot_sdk_request_timeline(
        df_sdk_timeline,
        title="SDK requests per minute (from log start)",
    )
    plt.show()

## Top API Calls

In [ ]:
top_calls = df_sdk_request_response["method_url"].value_counts().head(20).reset_index()
top_calls.columns = ["method_url", "call_count"]
top_calls

## Sample Request/Response Pairs

In [ ]:
df_sdk_request_response.head(100)